# S&P 500 SEC 데이터 스파이크

총부채 D/E 임계값을 수익률과 무관한 횡단면 분포로 정할 수 있는지, 그리고 매출총이익률 적신호의 구조적 결측이 어느 정도인지 재현한다.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
companyfacts_debt = json.loads((ROOT / 'sp500_debt_results.json').read_text(encoding='utf-8'))
fsds_debt = json.loads((ROOT / 'fsds_debt_results_2026q1.json').read_text(encoding='utf-8'))
gross = json.loads((ROOT / 'sp500_gross_profit_results.json').read_text(encoding='utf-8'))
inline = json.loads((ROOT / 'inline_fallback_results.json').read_text(encoding='utf-8'))

In [ ]:
debt_summary = {
    'Companyfacts strict complete': companyfacts_debt['coverage']['complete_issuers'],
    'Companyfacts population': companyfacts_debt['coverage']['eligible_issuers'],
    'FSDS strict complete': fsds_debt['strict']['complete_issuers'],
    'FSDS population': fsds_debt['population_issuers'],
    'FSDS strict P90 (not valid)': round(fsds_debt['strict']['distribution']['p90'], 4),
    'Relaxed P90 sensitivity only': fsds_debt['reported_materiality_convention']['p90_rounded_two_decimals'],
    'CAT inline periods reconciled': inline['coverage']['narrative_matches'],
}
debt_summary

In [ ]:
gross_status = gross['status_counts']
gross_sector = [
    {
        'sector': row['sector'],
        'direct_coverage_pct': round(row['direct_coverage'] * 100, 1),
        'upper_bound_pct': round(row['upper_bound_coverage_before_scope_validation'] * 100, 1),
    }
    for row in gross['by_sector']
]
gross_status, gross_sector

In [ ]:
assert inline['coverage']['available'] == 8
assert inline['coverage']['narrative_matches'] == 8
assert fsds_debt['strict']['coverage_of_population'] < 0.90
energy = next(row for row in gross['by_sector'] if row['sector'] == 'Energy')
assert energy['direct_8_of_8'] == 0
'All notebook checks passed.'

## 결론

- CAT의 차원형 부채는 inline XBRL로 8/8 복원되지만, 엄격한 금융리스 포함 D/E는 전체 커버리지 기준을 충족하지 못한다.
- 완화안의 P90은 임계값이 아니라 민감도 값이다.
- 매출총이익률 결측은 섹터에 따라 구조적이므로 적신호를 `평가 불가`로 분리한다.